In [ ]:
!pip install -q langchain langchain-groq langchain-community sqlalchemy

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")  # add your key in Colab's secrets (key icon, left sidebar)

In [ ]:
import pandas as pd
import sqlite3

!wget -q -O players_raw.csv "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2024-25/players_raw.csv"
!wget -q -O teams.csv "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2024-25/teams.csv"
!wget -q -O merged_gw.csv "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2024-25/gws/merged_gw.csv"

pos_map = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD", 5: "MNG"}

teams = pd.read_csv("teams.csv")[[
    "id", "name", "short_name", "strength", "position", "played",
    "win", "draw", "loss", "points",
    "strength_attack_home", "strength_attack_away",
    "strength_defence_home", "strength_defence_away"
]].rename(columns={"id": "team_id", "name": "team_name", "position": "league_position"})

players = pd.read_csv("players_raw.csv")
players = players[[
    "id", "first_name", "second_name", "web_name", "team", "element_type",
    "now_cost", "total_points", "minutes", "goals_scored", "assists",
    "clean_sheets", "goals_conceded", "yellow_cards", "red_cards", "saves",
    "bonus", "bps", "influence", "creativity", "threat", "ict_index",
    "selected_by_percent", "form", "points_per_game", "starts"
]].rename(columns={"id": "player_id", "team": "team_id"})
players["position"] = players["element_type"].map(pos_map)
players = players.drop(columns=["element_type"])
players["now_cost"] = players["now_cost"] / 10.0

gw = pd.read_csv("merged_gw.csv")
gw = gw[[
    "element", "name", "position", "team", "GW", "round", "opponent_team",
    "was_home", "total_points", "minutes", "goals_scored", "assists",
    "clean_sheets", "goals_conceded", "yellow_cards", "red_cards",
    "bonus", "bps", "influence", "creativity", "threat", "ict_index",
    "expected_goals", "expected_assists", "value", "selected",
    "transfers_in", "transfers_out"
]].rename(columns={"element": "player_id", "name": "player_name", "team": "team_name"})

conn = sqlite3.connect("fpl.db")
teams.to_sql("teams", conn, if_exists="replace", index=False)
players.to_sql("players", conn, if_exists="replace", index=False)
gw.to_sql("gameweek_stats", conn, if_exists="replace", index=False)

for t in ["teams", "players", "gameweek_stats"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(t, "->", n, "rows")
conn.close()
print("fpl.db built ✅ — fully fetched via API/links, no manual files")

teams -> 20 rows
players -> 804 rows
gameweek_stats -> 27605 rows
fpl.db built ✅ — fully fetched via API/links, no manual files


In [ ]:
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

# connect to the db we built
db = SQLDatabase.from_uri("sqlite:///fpl.db")

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # much lower token usage, higher rate limits, good enough for SQL gen
    temperature=0
)

agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="tool-calling",
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    top_k=15
)

print("Agent ready ✅")
print(db.get_usable_table_names())

Agent ready ✅
['gameweek_stats', 'players', 'teams']


In [ ]:
response = agent.invoke({"input": "Which 5 forwards scored the most goals this season? Show name and goals."})
print(response["output"])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`
responded:   Then I should construct a query to get the answer.  Then I should check the query before executing it.  Then I should execute the query and return the answer.



gameweek_stats, players, teams
Invoking: `sql_db_schema` with `{'table_names': 'gameweek_stats, players, teams'}`



CREATE TABLE gameweek_stats (
	player_id INTEGER, 
	player_name TEXT, 
	position TEXT, 
	team_name TEXT, 
	"GW" INTEGER, 
	round INTEGER, 
	opponent_team INTEGER, 
	was_home INTEGER, 
	total_points INTEGER, 
	minutes INTEGER, 
	goals_scored INTEGER, 
	assists INTEGER, 
	clean_sheets INTEGER, 
	goals_conceded INTEGER, 
	yellow_cards INTEGER, 
	red_cards INTEGER, 
	bonus INTEGER, 
	bps INTEGER, 
	influence REAL, 
	creativity REAL, 
	threat REAL, 
	ict_index REAL, 
	expected_goals REAL, 
	expected_assists REAL, 
	value INTEGER, 
	selected INTEGER, 
	transfers_in INTEGER, 
	transfers_out INTEGER
)

/*
3 rows from game

In [ ]:
def ask(question: str):
    response = agent.invoke({"input": question})
    print("Q:", question)
    print("A:", response["output"])
    return response["output"]

In [ ]:
ask("Which team has the best attacking strength at home?")
ask("Which player had the highest single gameweek score this season, and in which gameweek?")



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`
responded:   Then I should construct a query to get the answer to the question.  Then I should check the query to make sure it is correct.  Then I should execute the query and return the answer.



gameweek_stats, players, teams
Invoking: `sql_db_schema` with `{'table_names': 'gameweek_stats, players, teams'}`



CREATE TABLE gameweek_stats (
	player_id INTEGER, 
	player_name TEXT, 
	position TEXT, 
	team_name TEXT, 
	"GW" INTEGER, 
	round INTEGER, 
	opponent_team INTEGER, 
	was_home INTEGER, 
	total_points INTEGER, 
	minutes INTEGER, 
	goals_scored INTEGER, 
	assists INTEGER, 
	clean_sheets INTEGER, 
	goals_conceded INTEGER, 
	yellow_cards INTEGER, 
	red_cards INTEGER, 
	bonus INTEGER, 
	bps INTEGER, 
	influence REAL, 
	creativity REAL, 
	threat REAL, 
	ict_index REAL, 
	expected_goals REAL, 
	expected_assists REAL, 
	value INTEGER, 
	selected INTEGER, 
	transfers_in INTEGER, 
	transfers_out INTEGER

'The player with the highest single gameweek score this season is Cole Palmer, and it was in Gameweek 6.'